# Scene 2 — Full-Frame Kamishibai Pipeline (NEW NOTEBOOK)

This notebook is a **new** version (does not modify your existing notebook) and is designed for Google Colab.

## What it does
- Input: one image
- Detect objects (person/animals/trees/sky/buildings/ground)
- Segment each object with SAM
- Assign depth layer (background/midground/foreground)
- Create **full-frame masks** (same resolution as input)
- Send full-frame base image + full-frame mask to OpenAI image edits
- Save layer images, masks, RGBA cutouts, and metadata JSON
- Visualize all steps + stacked preview

> Important: no object cropping is used for generation.


In [ ]:
# Install dependencies (Colab)
!pip -q install --upgrade pip
!pip -q install openai pillow numpy matplotlib opencv-python tqdm transformers accelerate timm scipy


In [ ]:
import os
import io
import json
import math
import base64
from dataclasses import dataclass, asdict
from typing import List, Dict, Tuple

import cv2
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from transformers import (
    pipeline,
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    SamProcessor,
    SamModel,
)
from openai import OpenAI

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


In [ ]:
# =============================
# CONFIG
# =============================
INPUT_IMAGE_PATH = '/content/data/input/input.png'
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')

assert os.path.exists(INPUT_IMAGE_PATH), f'Missing input image: {INPUT_IMAGE_PATH}'
assert OPENAI_API_KEY, 'Please set OPENAI_API_KEY in Colab environment.'

ROOT = '/content/data'
DIR_INPUT = os.path.join(ROOT, 'input')
DIR_MASKS = os.path.join(ROOT, 'masks')
DIR_LAYERS = os.path.join(ROOT, 'layers')
DIR_DEBUG = os.path.join(ROOT, 'debug')
for d in [DIR_INPUT, DIR_MASKS, DIR_LAYERS, DIR_DEBUG]:
    os.makedirs(d, exist_ok=True)

# Detection classes (semantic)
DETECTION_LABELS = [
    'person', 'deer', 'animal', 'tree', 'forest', 'building', 'house',
    'sky', 'cloud', 'mountain', 'ground', 'grass', 'road', 'water'
]
DETECTION_THRESHOLD = 0.22
NMS_IOU = 0.45
MIN_MASK_AREA_FRAC = 0.001

# One fixed prompt for all layers (style consistency)
STYLE_PROMPT = (
    'Japanese paper theater (kamishibai) paper-cut style. '
    'Flat colors, simplified silhouettes, clean cut-paper edges, layered composition, '
    'minimal texture, cohesive limited palette, no photorealism. '
    'Only render the masked subject clearly. Keep non-masked regions simple and unobtrusive. '
    'Preserve the original full-frame composition and scale.'
)

OPENAI_MODEL = 'gpt-image-1'
OPENAI_BG_COLOR = 245  # neutral background value for base image

print('Using prompt:', STYLE_PROMPT)


In [ ]:
# =============================
# HELPERS
# =============================
def load_rgb(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('RGB'))

def save_png(arr: np.ndarray, path: str):
    Image.fromarray(arr).save(path)

def clip_box_xyxy(box, w, h):
    x1, y1, x2, y2 = map(float, box)
    x1 = int(np.clip(x1, 0, w - 1))
    y1 = int(np.clip(y1, 0, h - 1))
    x2 = int(np.clip(x2, 0, w - 1))
    y2 = int(np.clip(y2, 0, h - 1))
    if x2 <= x1: x2 = min(w - 1, x1 + 1)
    if y2 <= y1: y2 = min(h - 1, y1 + 1)
    return [x1, y1, x2, y2]

def nms(boxes, scores, iou_threshold=0.45):
    boxes = np.array(boxes, dtype=np.float32)
    scores = np.array(scores, dtype=np.float32)
    if len(boxes) == 0:
        return []
    order = np.argsort(scores)[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        xx1 = np.maximum(boxes[i, 0], boxes[order[1:], 0])
        yy1 = np.maximum(boxes[i, 1], boxes[order[1:], 1])
        xx2 = np.minimum(boxes[i, 2], boxes[order[1:], 2])
        yy2 = np.minimum(boxes[i, 3], boxes[order[1:], 3])
        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
        area_j = (boxes[order[1:], 2] - boxes[order[1:], 0]) * (boxes[order[1:], 3] - boxes[order[1:], 1])
        iou = inter / (area_i + area_j - inter + 1e-6)
        order = order[np.where(iou <= iou_threshold)[0] + 1]
    return keep

def clean_mask(mask01: np.ndarray, min_area: int) -> np.ndarray:
    m = (mask01 > 0).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    out = np.zeros_like(m)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            out[labels == i] = 1
    kernel = np.ones((5,5), np.uint8)
    out = cv2.morphologyEx(out, cv2.MORPH_OPEN, kernel)
    out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, kernel)
    return out.astype(np.uint8)

def draw_detections(img: np.ndarray, dets: List[Dict]) -> np.ndarray:
    out = Image.fromarray(img.copy())
    draw = ImageDraw.Draw(out)
    for d in dets:
        x1,y1,x2,y2 = d['bbox']
        draw.rectangle([x1,y1,x2,y2], outline=(255,0,0), width=3)
        draw.text((x1+3, max(0,y1-12)), f"{d['label']} {d['score']:.2f}", fill=(255,0,0))
    return np.array(out)

def to_rgba_cutout(rgb: np.ndarray, mask01: np.ndarray) -> np.ndarray:
    return np.dstack([rgb, (mask01*255).astype(np.uint8)])


In [ ]:
# =============================
# MODELS (loaded once)
# =============================

# Florence equivalent detector: GroundingDINO zero-shot
DET_MODEL_ID = 'IDEA-Research/grounding-dino-tiny'
det_processor = AutoProcessor.from_pretrained(DET_MODEL_ID)
det_model = AutoModelForZeroShotObjectDetection.from_pretrained(DET_MODEL_ID).to(DEVICE)

# SAM
sam_processor = SamProcessor.from_pretrained('facebook/sam-vit-base')
sam_model = SamModel.from_pretrained('facebook/sam-vit-base').to(DEVICE)

# Depth
depth_pipe = pipeline('depth-estimation', model='Intel/dpt-large', device=0 if DEVICE=='cuda' else -1)

# OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print('Models loaded.')


In [ ]:
# =============================
# 1) Object Detection
# =============================
def detect_objects(image_rgb: np.ndarray, labels: List[str], threshold=0.22) -> List[Dict]:
    image_pil = Image.fromarray(image_rgb)
    # GroundingDINO expects a caption-like prompt
    text_prompt = '. '.join(labels) + '.'

    inputs = det_processor(images=image_pil, text=text_prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = det_model(**inputs)

    results = det_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        box_threshold=threshold,
        text_threshold=0.2,
        target_sizes=[image_pil.size[::-1]]
    )[0]

    h, w = image_rgb.shape[:2]
    dets = []
    for box, score, text_label in zip(results['boxes'], results['scores'], results['labels']):
        label = text_label.strip().lower()
        b = clip_box_xyxy(box.detach().cpu().numpy().tolist(), w, h)
        dets.append({'label': label, 'score': float(score.item()), 'bbox': b})

    # NMS
    keep = nms([d['bbox'] for d in dets], [d['score'] for d in dets], iou_threshold=NMS_IOU)
    return [dets[i] for i in keep]


In [ ]:
# =============================
# 2) Segmentation (SAM)
# =============================
def segment_objects(image_rgb: np.ndarray, detections: List[Dict]) -> List[Dict]:
    if not detections:
        return []
    image_pil = Image.fromarray(image_rgb)
    input_boxes = [[d['bbox'] for d in detections]]

    inputs = sam_processor(images=image_pil, input_boxes=input_boxes, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = sam_model(**inputs)

    post_masks = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs['original_sizes'].cpu(),
        inputs['reshaped_input_sizes'].cpu(),
    )[0]

    h, w = image_rgb.shape[:2]
    min_area = int(h * w * MIN_MASK_AREA_FRAC)

    out = []
    for i, d in enumerate(detections):
        candidates = post_masks[i].numpy()   # [3, H, W]
        ious = outputs.iou_scores[0, i].detach().cpu().numpy()
        best = int(np.argmax(ious))
        mask01 = (candidates[best] > 0).astype(np.uint8)
        mask01 = clean_mask(mask01, min_area=min_area)
        if mask01.sum() < min_area:
            continue
        out.append({**d, 'mask': mask01})
    return out


In [ ]:
# =============================
# 3) Depth + Layer Assignment
# =============================
def estimate_depth(image_rgb: np.ndarray) -> np.ndarray:
    d = depth_pipe(Image.fromarray(image_rgb))['depth']
    d = np.array(d).astype(np.float32)
    d = cv2.resize(d, (image_rgb.shape[1], image_rgb.shape[0]), interpolation=cv2.INTER_CUBIC)
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)
    return d

def assign_layers(objs: List[Dict], depth_map: np.ndarray) -> List[Dict]:
    if not objs:
        return []
    med = []
    for o in objs:
        vals = depth_map[o['mask'] > 0]
        med.append(float(np.median(vals)) if len(vals) else 0.5)

    order = np.argsort(med)  # near -> far (for this normalized map)
    n = len(objs)
    out = [None] * n
    for rank, idx in enumerate(order):
        if rank < max(1, n // 3):
            layer = 'foreground'
        elif rank < max(2, (2*n)//3):
            layer = 'midground'
        else:
            layer = 'background'
        item = dict(objs[idx])
        item['depth_score'] = med[idx]
        item['layer'] = layer
        out[idx] = item
    return out


In [ ]:
# =============================
# 4+5) Full-frame OpenAI edit
# =============================
def _nearest_openai_size(w: int, h: int) -> Tuple[int, int]:
    # Conservative common sizes; if your account supports arbitrary sizes, you can bypass this.
    supported = [(1024,1024), (1536,1024), (1024,1536)]
    target_ar = w / max(1, h)
    best = min(supported, key=lambda s: abs((s[0]/s[1]) - target_ar))
    return best

def build_edit_assets(mask01: np.ndarray, base_gray=245):
    h, w = mask01.shape
    base = np.full((h, w, 3), base_gray, dtype=np.uint8)

    # Transparent where edits are allowed (object area)
    alpha = np.where(mask01 > 0, 0, 255).astype(np.uint8)
    mask_rgba = np.dstack([np.zeros((h,w,3), dtype=np.uint8), alpha])
    return Image.fromarray(base, 'RGB'), Image.fromarray(mask_rgba, 'RGBA')

def openai_generate_fullframe(mask01: np.ndarray, prompt: str) -> np.ndarray:
    base_pil, mask_pil = build_edit_assets(mask01, base_gray=OPENAI_BG_COLOR)
    orig_w, orig_h = base_pil.size

    # Improve reliability for model size constraints
    req_w, req_h = _nearest_openai_size(orig_w, orig_h)
    if (req_w, req_h) != (orig_w, orig_h):
        base_send = base_pil.resize((req_w, req_h), Image.Resampling.LANCZOS)
        mask_send = mask_pil.resize((req_w, req_h), Image.Resampling.NEAREST)
    else:
        base_send, mask_send = base_pil, mask_pil

    b_img = io.BytesIO(); base_send.save(b_img, format='PNG'); b_img.seek(0)
    b_msk = io.BytesIO(); mask_send.save(b_msk, format='PNG'); b_msk.seek(0)

    result = client.images.edit(
        model=OPENAI_MODEL,
        image=b_img,
        mask=b_msk,
        prompt=prompt,
        size=f'{req_w}x{req_h}',
    )

    out = np.array(Image.open(io.BytesIO(base64.b64decode(result.data[0].b64_json))).convert('RGB'))
    if (req_w, req_h) != (orig_w, orig_h):
        out = np.array(Image.fromarray(out).resize((orig_w, orig_h), Image.Resampling.LANCZOS))
    return out

# Swap point for providers:
# def generate_layer(mask01, prompt, provider='openai'):
#     if provider == 'openai':
#         return openai_generate_fullframe(mask01, prompt)
#     elif provider == 'gemini':
#         raise NotImplementedError('Implement Gemini full-frame edit using same mask logic.')


In [ ]:
# =============================
# 6+7) Run end-to-end + save
# =============================
image_rgb = load_rgb(INPUT_IMAGE_PATH)
H, W = image_rgb.shape[:2]

# 1 detect
detections = detect_objects(image_rgb, DETECTION_LABELS, threshold=DETECTION_THRESHOLD)
print('detections:', len(detections))

# 2 segment
segmented = segment_objects(image_rgb, detections)
print('segmented:', len(segmented))

# 3 depth+layers
depth_map = estimate_depth(image_rgb)
layered = assign_layers(segmented, depth_map)

metadata = {
    'input_image': INPUT_IMAGE_PATH,
    'image_size': {'width': W, 'height': H},
    'style_prompt': STYLE_PROMPT,
    'objects': []
}

for i, obj in enumerate(tqdm(layered, desc='Generating layers')):
    label_safe = obj['label'].replace(' ', '_')

    mask_path = os.path.join(DIR_MASKS, f'{i:03d}_{label_safe}_mask.png')
    layer_path = os.path.join(DIR_LAYERS, f'{i:03d}_{label_safe}_{obj["layer"]}.png')
    rgba_path = os.path.join(DIR_LAYERS, f'{i:03d}_{label_safe}_{obj["layer"]}_rgba.png')

    mask01 = obj['mask'].astype(np.uint8)
    save_png((mask01 * 255).astype(np.uint8), mask_path)

    # Full-frame generation from full-frame mask
    stylized = openai_generate_fullframe(mask01, STYLE_PROMPT)
    save_png(stylized, layer_path)

    rgba = to_rgba_cutout(stylized, mask01)
    save_png(rgba, rgba_path)

    metadata['objects'].append({
        'id': i,
        'label': obj['label'],
        'layer': obj['layer'],
        'bbox': obj['bbox'],
        'depth_score': obj['depth_score'],
        'mask_path': mask_path,
        'layer_image_path': layer_path,
        'rgba_path': rgba_path,
    })

meta_path = os.path.join(ROOT, 'metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('saved metadata ->', meta_path)
print('total layers ->', len(metadata['objects']))


In [ ]:
# =============================
# 8+9) Visualizations
# =============================
boxed = draw_detections(image_rgb, detections)
plt.figure(figsize=(10,8)); plt.imshow(boxed); plt.axis('off'); plt.title('Detected Boxes'); plt.show()

n = len(metadata['objects'])
cols = 4
rows = max(1, math.ceil(n / cols))

plt.figure(figsize=(4*cols, 4*rows))
for i, o in enumerate(metadata['objects']):
    m = np.array(Image.open(o['mask_path']).convert('L'))
    plt.subplot(rows, cols, i+1)
    plt.imshow(m, cmap='gray')
    plt.title(f"{o['id']} {o['label']} ({o['layer']})")
    plt.axis('off')
plt.tight_layout(); plt.show()

plt.figure(figsize=(4*cols, 4*rows))
for i, o in enumerate(metadata['objects']):
    lay = np.array(Image.open(o['layer_image_path']).convert('RGB'))
    plt.subplot(rows, cols, i+1)
    plt.imshow(lay)
    plt.title(f"{o['id']} {o['label']} ({o['layer']})")
    plt.axis('off')
plt.tight_layout(); plt.show()

# stacked preview
stack = np.full_like(image_rgb, OPENAI_BG_COLOR)
order_map = {'background':0, 'midground':1, 'foreground':2}
for o in sorted(metadata['objects'], key=lambda x: order_map.get(x['layer'], 1)):
    lay = np.array(Image.open(o['layer_image_path']).convert('RGB'))
    m = (np.array(Image.open(o['mask_path']).convert('L')) > 0)
    stack[m] = lay[m]

plt.figure(figsize=(10,8)); plt.imshow(stack); plt.axis('off'); plt.title('Stacked Preview'); plt.show()

save_png(boxed, os.path.join(DIR_DEBUG, 'detected_boxes.png'))
save_png(stack, os.path.join(DIR_DEBUG, 'stacked_preview.png'))
print('debug saved ->', DIR_DEBUG)


## Notes on swapping models/providers
- Detection function `detect_objects(...)`: swap GroundingDINO for Florence / OWL-V2 if desired.
- Segmentation function `segment_objects(...)`: swap SAM base for SAM2.
- Depth function `estimate_depth(...)`: swap DPT for MiDaS / Depth-Anything.
- Generation function `openai_generate_fullframe(...)`: replace with Gemini edit API while keeping:
  1) full-frame base,
  2) full-frame mask,
  3) same prompt for all objects.
